# Register Model

## Notebook Overview

- Start Execution
- Install and Import Libraries
- Configure Settings
- Model Service Registration to MLFlow

In [1]:
%%time

%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.
CPU times: user 15 ms, sys: 16.8 ms, total: 31.8 ms
Wall time: 1.3 s


In [2]:
MIN_TOTAL_RAM_GB = 16
MIN_TOTAL_VRAM_GB = 8


from ai_studio_blueprint_kit.memory_guard import run_memory_check_notebook


run_memory_check_notebook(
    min_total_ram_gb=MIN_TOTAL_RAM_GB,
    min_total_vram_gb=MIN_TOTAL_VRAM_GB,
)

# Start Execution

In [3]:
import logging
import time

# Configure logger
logger: logging.Logger = logging.getLogger("register_model_logger")
logger.setLevel(logging.INFO)
logger.propagate = False  # Prevent duplicate logs from parent loggers

# Set formatter
formatter: logging.Formatter = logging.Formatter(
    fmt="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# Configure and attach stream handler
stream_handler: logging.StreamHandler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

In [4]:
start_time = time.time()  

logger.info("Notebook execution started.")

2026-04-16 21:37:11 - INFO - Notebook execution started.


# Install and Import Libraries

In [ ]:

# === Standard Library Imports ===
import os
import sys
import json
import warnings
from datetime import datetime
from pathlib import Path

# Define the relative path to the 'src' directory (two levels up from current working directory)
src_path = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add 'src' directory to system path for module imports (e.g., utils)
if src_path not in sys.path:
    sys.path.append(src_path)

# === Third-Party Imports ===
import numpy as np
import pandas as pd
import mlflow
from typing import List

# Import transformers from huggingface
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

#Import components of notebook
from core.extract_text.arxiv_search import ArxivSearcher
from core.generator.script_generator import ScriptGenerator
from core.analyzer.scientific_paper_analyzer import ScientificPaperAnalyzer

#import langchain libraries
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEndpoint
from langchain_core.callbacks import CallbackManager, StreamingStdOutCallbackHandler

# === Project-Specific Imports (from src.utils) ===
from src.utils import (
    load_config,
    load_secrets,
    load_secrets_to_env,
    configure_proxy,
    initialize_llm,
    configure_hf_cache
)

# Import MLflow components for model signature
from mlflow.models import ModelSignature
from mlflow.types import Schema, ColSpec, DataType, ParamSpec, ParamSchema

# Import the Logger for models-from-code pattern
from src.mlflow import Logger

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from IPython import get_ipython

# Configure Settings

In [6]:
# ------------------------ Suppress Verbose Logs ------------------------
warnings.filterwarnings("ignore")

In [7]:
# In case you just want to run this cell without the rest of the notebook 
# (you still need to install the requirements and run the import block), run the following block:
CONFIG_PATH = "../configs/config.yaml"
SECRETS_PATH = "../configs/secrets.yaml"
LOCAL_MODEL_PATH = "/home/jovyan/datafabric/meta-llama3.1-8b-Q8/Meta-Llama-3.1-8B-Instruct-Q8_0.gguf"

# Define demo folder path
DEMO_FOLDER = "../demo"


In [8]:
# Load secrets from secrets.yaml file (if it exists) into environment
if Path(SECRETS_PATH).exists():
    load_secrets_to_env(SECRETS_PATH)
else:
    print(f"No secrets file found at {SECRETS_PATH}; relying on preexisting environment")

# Retrieve secrets from environment
try:
    secrets = load_secrets()
except ValueError:
    secrets = {}

# Load configuration and secrets
config = load_config(CONFIG_PATH)

print("✅ Configuration loaded successfully")
print("✅ Secrets loaded successfully")

No secrets file found at ../configs/secrets.yaml; relying on preexisting environment
✅ Configuration loaded successfully
✅ Secrets loaded successfully


# Model Service Registration to MLFlow

In this section, we implement the **Model Service**, a REST API responsible for serving the language model. The API is automatically documented using Swagger (via FastAPI), enabling interactive testing and clear documentation of the endpoints.

## Text Generation Service

This section demonstrates how to use our TextGenerationService from the src/service directory. This approach improves code organization by separating the service implementation from the notebook, making it easier to maintain and update.

In [9]:
%%time

mlflow.set_tracking_uri('/phoenix/mlflow')
# Set up the MLflow experiment
mlflow.set_experiment("Text-Generation-service")

# Get model path from config
model_source = config.get("model_source", "local")
model_path = config.get("model_path", LOCAL_MODEL_PATH)

# Check if the model file exists
if model_path and not os.path.exists(model_path):
    logger.info(f"⚠️ Warning: Model file not found at {model_path}. Please verify the path.")

#Only logs the model path in the case where it is local
if config["model_source"] == "local":
    model_path = model_path
else:
    model_path = None

# Define MLflow signature with input and output schemas only
# All parameters are passed as DataFrame columns (single row), not as lists
input_schema = Schema([
    ColSpec(DataType.string, "query"),
    ColSpec(DataType.long, "max_results"),
    ColSpec(DataType.long, "chunk_size"),
    ColSpec(DataType.long, "chunk_overlap"),
    ColSpec(DataType.boolean, "do_extract"),
    ColSpec(DataType.boolean, "do_analyze"),
    ColSpec(DataType.boolean, "do_generate"),
    ColSpec(DataType.string, "analysis_prompt"),
    ColSpec(DataType.string, "generation_prompt"),
])

# Output schema defines the returned DataFrame structure
output_schema = Schema([
    ColSpec(DataType.string, "extracted_papers"),
    ColSpec(DataType.string, "script"),
])

# Create signature without params - all data goes through inputs as single-row DataFrame
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

# Log and register the model using Logger.log_model (models-from-code pattern)
with mlflow.start_run(run_name="Script-Generation") as run:
    # Use Logger.log_model to register the model following the new pattern
    Logger.log_model(
        signature=signature,
        artifact_path="script_generation_model",
        config_path=CONFIG_PATH,
        data_path=None,  # No data directory needed for text generation
        secrets_dict=secrets if secrets else None,
        model_path=model_path,
        demo_folder=DEMO_FOLDER
    )
    
    # Register the model in MLflow Model Registry
    model_uri = f"runs:/{run.info.run_id}/script_generation_model"
    mlflow.register_model(model_uri=model_uri, name="Script-Generation-Service")
    logger.info(f"✅ Model registered successfully with run ID: {run.info.run_id}")

2026/04/16 21:37:19 INFO mlflow.tracking.fluent: Experiment with name 'Text-Generation-service' does not exist. Creating a new experiment.
Successfully registered model 'Script-Generation-Service'.
2026/04/16 21:41:37 WARNING mlflow.tracking._model_registry.fluent: Run with id 3da9b8009ec4435993382bec7e577f7a has no artifacts at artifact path 'script_generation_model', registering model based on models:/m-737553ed35374d1096f74ceddc21f99e instead
Created version '1' of model 'Script-Generation-Service'.
2026-04-16 21:41:38 - INFO - ✅ Model registered successfully with run ID: 3da9b8009ec4435993382bec7e577f7a


CPU times: user 1.6 s, sys: 23.4 s, total: 25 s
Wall time: 4min 19s


In [10]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")

2026-04-16 21:41:38 - INFO - ⏱️ Total execution time: 4m 27.65s


In [1]:
status = "Notebook execution completed successfully"
print(f"Message: {status}")

Message: Notebook execution completed successfully


In [ ]:
app = get_ipython()
app.kernel.do_shutdown(restart=False)

Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).